# P2PNet Training вЂ” Sanash Almaty Dataset
Dataset must be uploaded to Kaggle as a dataset and mounted at `/kaggle/input/sanash-almaty/`

Expected structure:
```
/kaggle/input/sanash-almaty/
  p2pnet_almaty_dataset_block_stratified/
    images/nonempty/   <- all 495 images
    gt/                <- 495 .npy point files (same basename)
    train.txt          <- one image filename per line
    val.txt
```

In [ ]:
# в”Ђв”Ђ Cell 1: Install deps & clone P2PNet 
import subprocess, os

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(result.stdout[-500:] or 'вњ“')
    return result

run('pip install timm scipy --quiet')

if not os.path.exists('/kaggle/working/P2PNet'):
    run('git clone https://github.com/TencentARC/P2PNet.git /kaggle/working/P2PNet')

os.chdir('/kaggle/working/P2PNet')
print('CWD:', os.getcwd())

In [ ]:
import os
DATA_ROOT = "/kaggle/input/sanash-almaty"
print(os.listdir(DATA_ROOT))

In [ ]:
import zipfile, shutil

WORK_DIR = '/kaggle/working/dataset'
os.makedirs(WORK_DIR, exist_ok=True)

for zip_name, extract_to in [('images.zip', f'{WORK_DIR}/images/nonempty'), ('gt.zip', f'{WORK_DIR}/gt')]:
    zip_path = f'{DATA_ROOT}/{zip_name}'
    if os.path.exists(zip_path):
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f'Extracted {zip_name} → {extract_to}')

for f in ['train.txt', 'val.txt']:
    shutil.copy(f'{DATA_ROOT}/{f}', f'{WORK_DIR}/{f}')

DATA_ROOT = WORK_DIR
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
# Cell 4: Verify GT format
import glob
import numpy as np
from pathlib import Path

gts = glob.glob(f'{DATA_ROOT}/gt/*.npy')
print(f'GT files: {len(gts)}')
sample_gt = gts[0]
pts = np.load(sample_gt)
print(f'Sample GT: {Path(sample_gt).name}')
print(f'  Shape: {pts.shape}, dtype: {pts.dtype}')
print(f'  First 3 points: {pts[:3]}')
# Expected: (N, 2) float array of [x, y] coordinates
# If shape is (N, 3), we strip the 3rd col below in the dataset class

In [ ]:
import os
os.makedirs('crowd_datasets/ALMATY', exist_ok=True)

dataset_code = '''
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image


class AlmatyTransit(Dataset):
    """
    Custom dataset for Sanash Almaty bus interior footage.
    Expects:
      data_root/images/nonempty/<name>.jpg
      data_root/gt/<name>.npy  вЂ” shape (N, 2) or (N, 3); x,y coords in pixels
      data_root/train.txt or val.txt вЂ” one filename (with or without extension) per line
    """
    def __init__(self, data_root, transform=None, train=False, patch=False, flip=False):
        self.data_root  = data_root
        self.transform  = transform
        self.train      = train
        self.patch      = patch
        self.flip       = flip

        split_file = os.path.join(data_root, 'train.txt' if train else 'val.txt')
        with open(split_file) as f:
            names = [l.strip() for l in f if l.strip()]

        # Resolve full paths
        self.samples = []
        img_dir = os.path.join(data_root, 'images', 'nonempty')
        gt_dir  = os.path.join(data_root, 'gt')

        for name in names:
            stem = os.path.splitext(os.path.basename(name))[0]
            # Try jpg then png
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    break
            else:
                # name might already be a full path
                img_path = name if os.path.exists(name) else None
            gt_path = os.path.join(gt_dir, stem + '.npy')
            if img_path and os.path.exists(gt_path):
                self.samples.append((img_path, gt_path))
            else:
                print(f'[WARN] Missing pair for {stem}: img={img_path}, gt={gt_path}')

        print(f'AlmatyTransit ({"train" if train else "val"}): {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, gt_path = self.samples[idx]

        img = Image.open(img_path).convert('RGB')
        pts = np.load(gt_path).astype(np.float32)

        # Normalise to (N, 2)
        if pts.ndim == 1:
            pts = pts.reshape(-1, 2)
        pts = pts[:, :2]   # drop extra cols if any

        # Optional horizontal flip
        if self.train and self.flip and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            w   = img.width
            pts[:, 0] = w - pts[:, 0]

        # Optional random crop patch (P2PNet style)
        if self.train and self.patch:
            img, pts = self._random_crop(img, pts)

        if self.transform:
            img = self.transform(img)

        target = {
            'point':  torch.tensor(pts,                         dtype=torch.float32),
            'labels': torch.ones(len(pts),                      dtype=torch.long),
        }
        return img, target

    def _random_crop(self, img, pts, crop_size=512):
        w, h = img.size
        crop_size = min(crop_size, w, h)
        x0 = random.randint(0, w - crop_size)
        y0 = random.randint(0, h - crop_size)
        img = img.crop((x0, y0, x0 + crop_size, y0 + crop_size))
        # Keep only points inside crop
        mask = (
            (pts[:, 0] >= x0) & (pts[:, 0] < x0 + crop_size) &
            (pts[:, 1] >= y0) & (pts[:, 1] < y0 + crop_size)
        )
        pts  = pts[mask]
        pts[:, 0] -= x0
        pts[:, 1] -= y0
        return img, pts


def build(image_set, args):
    return AlmatyTransit(
        data_root=args.data_root,
        transform=None,          # transform injected by P2PNet training loop
        train=(image_set == 'train'),
        patch=args.use_patch if hasattr(args, 'use_patch') else False,
        flip=True if image_set == 'train' else False,
    )
'''

with open('crowd_datasets/ALMATY/ALMATY.py', 'w') as f:
    f.write(dataset_code)

with open('crowd_datasets/ALMATY/__init__.py', 'w') as f:
    f.write('')

print('Dataset class written.')

In [ ]:
# в”Ђв”Ђ Cell 5: Patch crowd_datasets/__init__.py to register ALMATY 
init_path = 'crowd_datasets/__init__.py'
with open(init_path) as f:
    src = f.read()

if 'ALMATY' not in src:
    # Append import and registration
    patch = """
from .ALMATY.ALMATY import build as build_almaty
"""
    src = patch + src
    # Also patch the build() dispatcher
    src = src.replace(
        "raise ValueError(f'dataset {dataset_file} not supported')",
        "if dataset_file == 'ALMATY': return build_almaty(image_set, args)\n    raise ValueError(f'dataset {dataset_file} not supported')"
    )
    with open(init_path, 'w') as f:
        f.write(src)
    print('Patched crowd_datasets/__init__.py')
else:
    print('Already patched.')

# Show result
with open(init_path) as f:
    print(f.read())

In [ ]:
# в”Ђв”Ђ Cell 6: Smoke-test dataset loads correctly 
import sys
sys.path.insert(0, '/kaggle/working/P2PNet')

from crowd_datasets.ALMATY.ALMATY import AlmatyTransit
import torchvision.transforms as T

transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = AlmatyTransit(DATA_ROOT, transform=transform, train=True,  flip=True)
val_ds   = AlmatyTransit(DATA_ROOT, transform=transform, train=False)

img, target = train_ds[0]
print(f'Image tensor shape : {img.shape}')
print(f'Points shape       : {target["point"].shape}')
print(f'Labels shape       : {target["labels"].shape}')
print(f'People in frame    : {len(target["point"])}')

In [ ]:
#  Cell 7: Download VGG16 backbone weights 
# P2PNet uses VGG16 with BN pretrained on ImageNet.
# torchvision downloads automatically on first use, but let's pre-fetch.
import torchvision.models as models
vgg = models.vgg16_bn(pretrained=True)
del vgg
print('VGG16-BN weights cached.')

In [ ]:
#  Cell 8: TRAIN 
# Hyperparameters tuned for 390 training images on a Kaggle T4.
# --epochs 500 with lr_drop at 400 gives a good starting point.
# Expect MAE < 3 on this clean indoor dataset after full run.

import subprocess, time

cmd = [
    'python', 'train.py',
    '--data_root',    DATA_ROOT,
    '--dataset_file', 'ALMATY',
    '--backbone',     'vgg16_bn',
    '--epochs',       '500',
    '--lr_drop',      '400',
    '--lr',           '0.0001',
    '--batch_size',   '8',          # safe for T4 16GB with 720p frames
    '--weight_decay', '0.0001',
    '--num_workers',  '2',
    '--output_dir',   '/kaggle/working/logs/almaty',
    '--row',          '2',          # 2Г—2 = 4 anchor points per cell
    '--line',         '2',
    '--eval_freq',    '5',          # validate every 5 epochs
]

os.makedirs('/kaggle/working/logs/almaty', exist_ok=True)

print('Starting training...')
print(' '.join(cmd))
print('в”Ђ' * 60)

start = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/kaggle/working/P2PNet'
)

for line in proc.stdout:
    print(line, end='')

proc.wait()
elapsed = time.time() - start
print(f'\nDone in {elapsed/60:.1f} min. Exit code: {proc.returncode}')

In [ ]:
# в”Ђв”Ђ Cell 9: Show best checkpoint metrics в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
import json, glob
from pathlib import Path

log_dir = Path('/kaggle/working/logs/almaty')
log_file = log_dir / 'log.txt'

if log_file.exists():
    lines = log_file.read_text().strip().split('\n')
    records = [json.loads(l) for l in lines if l.strip()]
    best = min(records, key=lambda r: r.get('test_mae', 999))
    print(f'Best epoch : {best.get("epoch")}')
    print(f'  MAE      : {best.get("test_mae", "N/A"):.4f}')
    print(f'  MSE      : {best.get("test_mse", "N/A"):.4f}')
else:
    print('log.txt not found вЂ” check training output above.')

# List saved checkpoints
ckpts = sorted(log_dir.glob('*.pth'))
print('\nCheckpoints saved:')
for c in ckpts:
    print(f'  {c.name}  ({c.stat().st_size/1e6:.1f} MB)')

In [ ]:
# в”Ђв”Ђ Cell 10: Quick visual sanity check on 4 val images в”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђв”Ђ
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pathlib import Path

sys.path.insert(0, '/kaggle/working/P2PNet')
from models import build_model
import argparse

# Find best checkpoint
log_dir = Path('/kaggle/working/logs/almaty')
best_ckpt = log_dir / 'best_mae.pth'
if not best_ckpt.exists():
    ckpts = sorted(log_dir.glob('*.pth'))
    best_ckpt = ckpts[-1] if ckpts else None

if best_ckpt is None:
    print('No checkpoint found.')
else:
    print(f'Loading: {best_ckpt}')

    # Minimal args object
    args = argparse.Namespace(
        backbone='vgg16_bn', row=2, line=2,
        num_classes=2, aux_loss=True,
        set_cost_class=1, set_cost_point=0.05,
        ce_loss_coef=1, point_loss_coef=0.0002,
        eos_coef=0.5, num_queries=700,
        hidden_dim=256, position_embedding='sine',
        lr_backbone=0.0,
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, _, _ = build_model(args)
    state = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(state['model'])
    model.to(device).eval()

    transform = T.Compose([
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    val_ds = AlmatyTransit(DATA_ROOT, transform=transform, train=False)
    indices = np.linspace(0, len(val_ds)-1, 4, dtype=int)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for ax, idx in zip(axes, indices):
        img_tensor, target = val_ds[idx]
        img_path, _ = val_ds.samples[idx]

        with torch.no_grad():
            outputs = model(img_tensor.unsqueeze(0).to(device))

        pred_points = outputs['pred_points'][0].cpu().numpy()
        pred_logits = outputs['pred_logits'][0].softmax(-1)[:, 1].cpu().numpy()

        # Filter confident predictions (threshold 0.5)
        keep = pred_logits > 0.5
        pred_pts = pred_points[keep]

        gt_pts = target['point'].numpy()
        gt_count   = len(gt_pts)
        pred_count = len(pred_pts)

        raw_img = Image.open(img_path).convert('RGB')
        ax.imshow(raw_img)
        if len(pred_pts):
            ax.scatter(pred_pts[:, 0], pred_pts[:, 1], c='red',  s=10, label='pred')
        if len(gt_pts):
            ax.scatter(gt_pts[:, 0],   gt_pts[:, 1],   c='lime', s=10, marker='+', label='gt')
        ax.set_title(f'GT={gt_count}  Pred={pred_count}', fontsize=10)
        ax.axis('off')

    handles = [
        mpatches.Patch(color='red',  label='Predicted'),
        mpatches.Patch(color='lime', label='Ground truth'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=2)
    plt.tight_layout()
    plt.savefig('/kaggle/working/val_predictions.png', dpi=120)
    plt.show()
    print('Saved: /kaggle/working/val_predictions.png')